In [ ]:
from pathlib import Path
import os
import pandas as pd
import random
#PATHING SCRIPT FOR EVERY EXCERSISE - PROJECT - WORKSPACE
# 1.Literal definition of route pathings(Every user of remote repository must config this pathing in order to find the local repository of his computer)
# My case:
ROOT = Path("/home/josu/Documentos/DataScienceCourse")

# We verify the existence before continue
if not ROOT.exists():
    raise FileNotFoundError(f"❌ The route {ROOT} doesn't exist. Check it.")

# 2. Fix the workspace
os.chdir(ROOT)
print(f"✅ Worskspace enabled!: {os.getcwd()}")

# 3. Define relative pathings to work properly
DATA_TABLES = ROOT / "data" / "raw" / "Tables"

# Let's verify our table folder:
if not DATA_TABLES.exists():
    # If it fails, monitorize the issue: 
    data_dir = ROOT / "data"
    if data_dir.exists():
        print(f"⚠️ the folder 'data' exists but it doesn't have any 'Tables'. 'data' content: {os.listdir(data_dir)}")
    else:
        print(f"⚠️ The folder 'data' doesn't exist on ROOT workspace: {os.listdir(ROOT)}")
    raise FileNotFoundError(f"❌ The tables route {DATA_TABLES} is missing.")

print(f"📂 Tables route detected: {DATA_TABLES}")


In [ ]:
# ==========================================================
# KNOWLEDGE SCRAPER
# ----------------------------------------------------------
# BLOCK 1 - HTML DOWNLOADER
#
# Purpose:
#     Download the fully rendered HTML source of a webpage.
#
# Responsibilities:
#     - Launch Chromium
#     - Load the webpage
#     - Return HTML
#
# It DOES NOT:
#     - Parse HTML
#     - Search tables
#     - Search lists
#     - Build DataFrames
#
'''
URL
 │
 ▼
download_html()
 │
 ▼
HTML
 │
 ▼
BeautifulSoup
 │
 ▼
Dispatcher
 │
 ├──────────────┐
 ▼              ▼
TABLE      HIERARCHY(Enumerated text list from a Semantic Root)
 │              │
 ▼              ▼
DataFrame   DataFrame
'''
# Input
# -----
# url : str
#     Webpage URL.
#
# Output
# ------
# html : str
#     Complete rendered HTML source.
# ==========================================================

from playwright.async_api import async_playwright


async def download_html(
    url: str,
    executable_path: str = "/usr/bin/chromium",
    headless: bool = True,
    wait_until: str = "domcontentloaded"
) -> str:

    pw = await async_playwright().start()

    browser = await pw.chromium.launch(
        executable_path=executable_path,
        headless=headless
    )

    page = await browser.new_page()

    await page.goto(
        url,
        wait_until=wait_until
    )

    html = await page.content()

    await browser.close()
    await pw.stop()

    return html

In [ ]:
# ==========================================================
# KNOWLEDGE SCRAPER
# ----------------------------------------------------------
# BLOCK 2 - SCRAPER DISPATCHER
#
# Purpose:
#     Control the complete scraping workflow.
#
# Responsibilities:
#     - Download HTML
#     - Create BeautifulSoup object
#     - Select the appropriate extractor
#     - Return a DataFrame
#
'''
URL
 │
 ▼
download_html()
 │
 ▼
HTML
 │
 ▼
BeautifulSoup
 │
 ▼
Dispatcher
 │
 ├──────────────┐
 ▼              ▼
TABLE      HIERARCHY
 │              │
 ▼              ▼
DataFrame   DataFrame
'''
# Input
# -----
# source : dict
#
# Output
# ------
# pandas.DataFrame
# ==========================================================

from bs4 import BeautifulSoup
import gc


async def scrape_source(source):

    # ---------------------------------------------
    # Download webpage
    # ---------------------------------------------

    html = await download_html(source["url"])

    # ---------------------------------------------
    # Build BeautifulSoup once
    # ---------------------------------------------

    soup = BeautifulSoup(html, "lxml")

    # ---------------------------------------------
    # Dispatcher
    # ---------------------------------------------

    source_type = source["type"].lower()

    if source_type == "table":

        df = extract_table_from_soup(
            soup=soup,
            expected_columns=source["columns"]
        )

    elif source_type == "hierarchy":

        df = extract_hierarchy_from_soup(
            soup=soup,
            root=source["root"]
        )

    else:

        raise ValueError(
            f"Unknown source type: {source_type}"
        )

    # ---------------------------------------------
    # Memory cleanup
    # ---------------------------------------------

    del html
    del soup

    gc.collect()

    return df

In [ ]:
# ==========================================================
# KNOWLEDGE SCRAPER
# ----------------------------------------------------------
# BLOCK 3 - TABLE EXTRACTOR
#
# Purpose:
#     Extract HTML tables whose headers contain the expected
#     column names.
#
'''
download_html()

↓

html

↓

BeautifulSoup()

↓

extract_table_from_soup()

↓

DataFrame
'''
# Responsibilities:
#     - Search tables
#     - Match headers
#     - Convert HTML table to DataFrame
#     - Keep only requested columns
#
# Input
# -----
# soup : BeautifulSoup
#
# expected_columns : list[str]
#
# Output
# ------
# pandas.DataFrame
# ==========================================================

from io import StringIO
import pandas as pd


def extract_table_from_soup(
    soup,
    expected_columns
):

    # ---------------------------------------------
    # Find every HTML table
    # ---------------------------------------------

    tables = soup.find_all("table")

    # ---------------------------------------------
    # Search matching table
    # ---------------------------------------------

    for table in tables:

        # Read table with pandas
        try:

            df = pd.read_html(
                StringIO(str(table))
            )[0]

        except Exception:
            continue

        # -----------------------------------------
        # Normalize column names
        # -----------------------------------------

        normalized_columns = []

        for col in df.columns:

            col = str(col)

            # Remove line breaks
            col = col.replace("\n", " ")

            # Collapse multiple spaces
            col = " ".join(col.split())

            normalized_columns.append(col)

        df.columns = normalized_columns

        # -----------------------------------------
        # Partial header matching
        # -----------------------------------------

        selected_columns = {}

        for expected in expected_columns:

            for real in df.columns:

                if expected.lower() in real.lower():

                    selected_columns[expected] = real
                    break

        # -----------------------------------------
        # Validate table
        # -----------------------------------------

        if len(selected_columns) == len(expected_columns):

            df = df[
                list(selected_columns.values())
            ]

            df.columns = expected_columns

            return df

    # ---------------------------------------------
    # No matching table
    # ---------------------------------------------

    raise ValueError(
        "No table matching the expected columns was found."
    )

# ==========================================================
# KNOWLEDGE SCRAPER
# ----------------------------------------------------------
# BLOCK 4 - HIERARCHY EXTRACTOR
#
# Purpose:
#     Extract hierarchical HTML structures and convert them
#     into a relational DataFrame.
#
# Responsibilities:
#     - Locate hierarchy root
#     - Detect categories
#     - Extract list items
#     - Build DataFrame
#
'''
download_html()

↓

BeautifulSoup()

↓

Dispatcher

↓

TABLE

↓

extract_table_from_soup()

↓

DataFrame

--------------------

HIERARCHY

↓

extract_hierarchy_from_soup()

↓

DataFrame
'''
# Input
# -----
# soup : BeautifulSoup
#
# root : str
#
# Output
# ------
# pandas.DataFrame
# ==========================================================

import pandas as pd


def extract_hierarchy_from_soup(
    soup,
    root
):

    # ---------------------------------------------
    # Locate hierarchy root
    # ---------------------------------------------

    root_heading = None

    for heading in soup.find_all(["h2", "h3"]):

        title = heading.get_text(" ", strip=True)

        if root.lower() in title.lower():

            root_heading = heading
            break

    if root_heading is None:

        raise ValueError(
            f'Hierarchy root "{root}" not found.'
        )

    # ---------------------------------------------
    # Walk through hierarchy
    # ---------------------------------------------

    records = []

    current_section = root

    current_category = None

    node = root_heading.find_next()

    while node:

        # Stop when next H2 starts
        if node.name == "h2" and node != root_heading:

            break

        # -----------------------------------------
        # New category
        # -----------------------------------------

        if node.name == "h3":

            current_category = node.get_text(
                " ",
                strip=True
            )

        # -----------------------------------------
        # Bullet list
        # -----------------------------------------

        elif node.name == "ul" and current_category:

            for li in node.find_all(
                "li",
                recursive=False
            ):

                text = li.get_text(
                    " ",
                    strip=True
                )

                records.append({

                    "Section": current_section,

                    "Category": current_category,

                    "Item": text,

                    "Description": ""

                })

        node = node.find_next()

    # ---------------------------------------------
    # Convert to DataFrame
    # ---------------------------------------------

    df = pd.DataFrame(records)

    return df

# ==========================================================
# KNOWLEDGE SCRAPER
# ----------------------------------------------------------
# BLOCK 4 - UNIVERSAL HIERARCHY EXTRACTOR (CONFIGURABLE)
#
# Purpose:
#     Extract hierarchical HTML structures from modern websites
#     or Wikipedia layouts where headings can vary in depth,
#     while separating item titles from descriptions.
#
# Responsibilities:
#     - Locate hierarchy root container / section dynamically
#     - Detect subsection headings (e.g., h3, h4, h5)
#     - Extract list items and split bold titles from descriptions
#     - Build structured relational DataFrame
# ==========================================================

import pandas as pd


def extract_hierarchy_from_soup(
    soup,
    root,
    header_tags=["h2", "h3", "h4", "h5", "h6"],
    list_tag="ul",
    item_tag="li",
    title_tag="b",
    container_tag="section"
):
    """
    Universal hierarchy extractor function matching original parameter signature.
    
    Parameters:
    - soup: BeautifulSoup parsed HTML object.
    - root (str): Target string/keyword to locate the root section heading.
    - header_tags (list): HTML tags considered as headings or subcategories.
    - list_tag (str): Container tag for lists (e.g., 'ul', 'ol').
    - item_tag (str): Individual list element tag (e.g., 'li').
    - title_tag (str): Internal tag highlighting the main item title (e.g., 'b', 'strong').
    - container_tag (str): Container section tag to isolate search scope.
    """

    # ---------------------------------------------
    # 1. Locate the hierarchy root heading
    # ---------------------------------------------
    root_heading = None

    # Search across all header tags to find the root section title
    for heading in soup.find_all(header_tags):
        title = heading.get_text(" ", strip=True)
        if root.lower() in title.lower():
            root_heading = heading
            break

    if root_heading is None:
        raise ValueError(
            f'Hierarchy root "{root}" not found.'
        )

    # Determine the parent section container if applicable
    root_section = root_heading.find_parent(container_tag)
    if not root_section:
        root_section = root_heading.parent

    # ---------------------------------------------
    # 2. Walk through hierarchy supporting dynamic structures
    # ---------------------------------------------
    records = []
    current_section = root
    current_category = None

    # Define the search scope using the container or heading element
    search_scope = root_section if root_section else root_heading

    # Look for both header tags and list tags within the section scope
    nodes = search_scope.find_all(header_tags + [list_tag])

    for node in nodes:
        text_val = node.get_text(" ", strip=True)

        # If node is a heading tag, update current category
        if node.name in header_tags:
            if root.lower() not in text_val.lower():
                current_category = text_val

        # If node is a list tag and we have an active category
        elif node.name == list_tag and current_category:
            for li in node.find_all(item_tag, recursive=False):

                # Extract bold title if present
                bold_tag = li.find(title_tag)

                if bold_tag:
                    item_title = bold_tag.get_text(" ", strip=True)
                    # Temporarily decompose the title tag to extract the description text
                    bold_tag.decompose()
                    description = li.get_text(" ", strip=True)
                    # Clean up leading punctuation/separators if left over
                    description = description.lstrip("—-: ").strip()
                else:
                    item_title = li.get_text(" ", strip=True)
                    description = ""

                records.append({
                    "Section": current_section,
                    "Category": current_category,
                    "Item": item_title,
                    "Description": description
                })

    # ---------------------------------------------
    # 3. Convert to DataFrame and Validate
    # ---------------------------------------------
    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError(f"No hierarchical items parsed under root '{root}'.")

    return df

# ==========================================================
# KNOWLEDGE SCRAPER
# ----------------------------------------------------------
# BLOCK 4 - ROBUST HIERARCHY EXTRACTOR (FAULT TOLERANT)
#
# Purpose:
#     Extract hierarchical HTML structures resiliently from 
#     complex pages where list items might have irregular formatting 
#     or mixed text nodes.
# ==========================================================

import pandas as pd


def extract_hierarchy_from_soup(
    soup,
    root,
    header_tags=["h2", "h3", "h4", "h5", "h6"],
    list_tag="ul",
    item_tag="li",
    title_tag="b",
    container_tag="section"
):
    """
    Robust hierarchy extractor handling irregular list layouts and missing bold tags.
    """

    # ---------------------------------------------
    # 1. Locate the hierarchy root heading
    # ---------------------------------------------
    root_heading = None

    for heading in soup.find_all(header_tags):
        title = heading.get_text(" ", strip=True)
        if root.lower() in title.lower():
            root_heading = heading
            break

    if root_heading is None:
        raise ValueError(f'Hierarchy root "{root}" not found.')

    # Determine the parent section container if applicable
    root_section = root_heading.find_parent(container_tag)
    if not root_section:
        root_section = root_heading.parent

    # ---------------------------------------------
    # 2. Walk through hierarchy with fault tolerance
    # ---------------------------------------------
    records = []
    current_section = root
    current_category = None

    search_scope = root_section if root_section else root_heading
    nodes = search_scope.find_all(header_tags + [list_tag])

    for node in nodes:
        text_val = node.get_text(" ", strip=True)

        # Update category on heading detection
        if node.name in header_tags:
            if root.lower() not in text_val.lower():
                current_category = text_val

        # Process list items securely
        elif node.name == list_tag and current_category:
            for li in node.find_all(item_tag, recursive=False):

                # Try to find the structured bold title element
                bold_tag = li.find(title_tag)

                if bold_tag:
                    item_title = bold_tag.get_text(" ", strip=True)
                    # Safely decompose to isolate description text
                    bold_tag.decompose()
                    description = li.get_text(" ", strip=True)
                    description = description.lstrip("—-: ").strip()
                else:
                    # Fallback logic for items lacking explicit <b> tags
                    full_text = li.get_text(" ", strip=True)
                    
                    # If there's a colon or dash, split title and description manually
                    if ":" in full_text:
                        parts = full_text.split(":", 1)
                        item_title = parts[0].strip()
                        description = parts[1].strip()
                    elif "—" in full_text:
                        parts = full_text.split("—", 1)
                        item_title = parts[0].strip()
                        description = parts[1].strip()
                    else:
                        item_title = full_text
                        description = ""

                records.append({
                    "Section": current_section,
                    "Category": current_category,
                    "Item": item_title,
                    "Description": description
                })

    # ---------------------------------------------
    # 3. Convert to DataFrame and Validate
    # ---------------------------------------------
    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError(f"No hierarchical items parsed under root '{root}'.")

    return df

In [ ]:
# ==========================================================
# KNOWLEDGE SCRAPER
# ----------------------------------------------------------
# BLOCK 4 - ROBUST HIERARCHY EXTRACTOR
#
# Purpose:
#     Extract hierarchical HTML structures and lists safely from 
#     modern Wikipedia layouts, utilizing positional order and 
#     intelligent fallback parsing to cleanly separate titles 
#     from descriptions into structured DataFrames.
#
# Responsibilities:
#     - Locate hierarchy root container / section dynamically
#     - Support deep subsection headings (h3, h4, h5)
#     - Extract list items via positional index matching
#     - Fallback robust text splitting for irregular list nodes
# ==========================================================

import re
import pandas as pd


def extract_hierarchy_from_soup(
    soup,
    root
):
    """
    Extracts hierarchical lists and descriptions under a specific root heading.
    """

    # ---------------------------------------------
    # 1. Locate the hierarchy root heading
    # ---------------------------------------------
    root_heading = None

    # Search across all header tags to find the root section title
    for heading in soup.find_all(["h2", "h3", "h4", "h5", "h6"]):
        title = heading.get_text(" ", strip=True)
        if root.lower() in title.lower():
            root_heading = heading
            break

    if root_heading is None:
        raise ValueError(
            f'Hierarchy root "{root}" not found.'
        )

    # Determine the parent section container if applicable
    root_section = root_heading.find_parent("section")
    if not root_section:
        root_section = root_heading.parent

    # ---------------------------------------------
    # 2. Walk through hierarchy with structural resilience
    # ---------------------------------------------
    records = []
    current_section = root
    current_category = None

    # Define the search scope using the section container
    search_scope = root_section if root_section else root_heading

    # Look for headers and lists within the section scope
    nodes = search_scope.find_all(["h3", "h4", "h5", "ul", "ol"])

    for node in nodes:
        text_val = node.get_text(" ", strip=True)

        # If node is a heading tag, update current category
        if node.name in ["h3", "h4", "h5"]:
            if root.lower() not in text_val.lower():
                current_category = text_val

        # If node is a list tag and we have an active category
        elif node.name in ["ul", "ol"] and current_category:
            list_items = node.find_all("li", recursive=False)

            for index, li in enumerate(list_items):
                full_text = li.get_text(" ", strip=True)

                # Try to find standard bold tag for the title
                bold_tag = li.find(["b", "strong"])

                if bold_tag:
                    item_title = bold_tag.get_text(" ", strip=True)
                    # Temporarily decompose the title tag to extract the remaining description
                    bold_tag.decompose()
                    description = li.get_text(" ", strip=True)
                    description = description.lstrip("—-: ").strip()
                else:
                    # Fallback pattern splitting for irregular nodes lacking bold tags
                    parts = re.split(r'[:—–-]', full_text, maxsplit=1)
                    if len(parts) > 1:
                        item_title = parts[0].strip()
                        description = parts[1].strip()
                    else:
                        item_title = full_text
                        description = ""

                records.append({
                    "Section": current_section,
                    "Category": current_category,
                    "Item": item_title,
                    "Description": description
                })

    # ---------------------------------------------
    # 3. Convert to DataFrame and Validate
    # ---------------------------------------------
    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError(f"No hierarchical items parsed under root '{root}'.")

    return df

In [ ]:
# ==========================================================
# KNOWLEDGE SCRAPER
# ----------------------------------------------------------
# BLOCK 5 - SOURCES CONFIGURATION
#
# Purpose:
#     Define every scraping task.
#
# Supported types:
#     - table
#     - hierarchy
# ==========================================================

SOURCES = [

    # ------------------------------------------------------
    # ISO Country Codes
    # ------------------------------------------------------
    {
        "name": "CountryCodes",

        "type": "table",

        "url": "https://es.wikipedia.org/wiki/ISO_3166-1_alfa-2",

        "columns": [

            "Código",
            "Nombre del país",
            "Año",
            "ccTLD",
            "ISO 3166-2",
            "Notas"

        ]
    },

    # ------------------------------------------------------
    # Official Languages
    # ------------------------------------------------------
    {
        "name": "OfficialLanguages",

        "type": "table",

        "url": "https://en.wikipedia.org/wiki/List_of_official_languages_by_country_and_territory",

        "columns": [

            "Country/Region",
            "Official language(s)",
            "National language(s)",
            "Regional language(s)",
            "Minority language(s)",
            "Widely spoken"

        ]
    },

    # ------------------------------------------------------
    # European Regions
    # ------------------------------------------------------
    {
        "name": "EuropeRegions",

        "type": "hierarchy",

        "url": "https://en.wikipedia.org/wiki/Regions_of_Europe",

        "root": "Geographical"

    }


]

In [ ]:
# ==========================================================
# KNOWLEDGE SCRAPER
# ----------------------------------------------------------
# BLOCK 6 - MAIN PIPELINE
#
# Purpose:
#     Execute every source sequentially.
# ==========================================================

datasets = {}

for source in SOURCES:

    print("=" * 60)
    print(f"Processing: {source['name']}")
    print("=" * 60)

    try:

        df = await scrape_source(source)

        datasets[source["name"]] = df

        print(f"Rows: {len(df)}")
        print(df.head())

    except Exception as e:

        print(f"ERROR -> {e}")

print("\nPipeline completed.")

In [ ]:
datasets.keys()

In [ ]:
datasets["CountryCodes"].info()

In [ ]:
datasets["EuropeRegions"].head()

In [ ]:
dfRegions = pd.DataFrame(datasets['EuropeRegions'])